
# Ingestão Iris, Produtos e Time Travel

Este notebook utiliza o Databricks Free Edition.

**Antes de executar:**
1. Carregue `iris_dataset.csv` como tabela `workspace.default.iris_dataset`.
2. Crie um Volume chamado `workspace` em `workspace.default`.
3. Envie `base_dim_produtos.csv` para esse Volume.

Nenhuma tabela existente será sobrescrita automaticamente.
A demonstração de Time Travel utiliza uma tabela separada.


In [ ]:
dbutils.widgets.text("catalogo", "workspace")
dbutils.widgets.text("esquema", "default")
dbutils.widgets.text("volume", "workspace")

catalogo = dbutils.widgets.get("catalogo").strip()
esquema = dbutils.widgets.get("esquema").strip()
volume = dbutils.widgets.get("volume").strip()

import re

for nome in [catalogo, esquema, volume]:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", nome):
        raise ValueError("Identificador invalido: " + nome)

tabela_iris = f"{catalogo}.{esquema}.iris_dataset"
tabela_produtos = f"{catalogo}.{esquema}.dim_produtos"
tabela_lab = f"{catalogo}.{esquema}.dim_produtos_time_travel_lab"

caminho_produtos = (
    f"/Volumes/{catalogo}/{esquema}/{volume}/base_dim_produtos.csv"
)

print("Tabela Iris:", tabela_iris)
print("Tabela Produtos:", tabela_produtos)
print("CSV no Volume:", caminho_produtos)


## 1. Consultar a tabela Iris

A tabela deve ter sido criada pela interface do Databricks.


In [ ]:
try:
    df_iris = spark.table(tabela_iris)
    print("Registros Iris:", df_iris.count())
    display(df_iris.limit(10))
except Exception as erro:
    print("Tabela Iris ainda nao encontrada.")
    print("Crie a tabela pela interface de ingestao.")
    print(type(erro).__name__)


## 2. Ingestão de produtos com tipagem explícita

Não utilizamos a sintaxe incorreta `SELECT id_produto STRING`.

O esquema é definido com `StructType`, utilizando PySpark.


In [ ]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DecimalType
)

esquema_produtos = StructType([
    StructField("id_produto", StringType(), False),
    StructField("nome_produto", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("preco_reais", DecimalType(10, 2), True)
])

try:
    df_produtos = (
        spark.read
        .option("header", "true")
        .option("sep", ",")
        .schema(esquema_produtos)
        .csv(caminho_produtos)
    )

    quantidade = df_produtos.count()

    print("Produtos encontrados:", quantidade)

    if quantidade != 15:
        raise ValueError(
            f"Esperados 15 produtos; encontrados {quantidade}"
        )

    display(df_produtos)

except Exception as erro:
    print("Leitura ainda nao concluida.")
    print("Confirme o upload do CSV para o Volume.")
    print(type(erro).__name__)
    df_produtos = None


## 3. Gravar a dimensão de produtos

Cria a tabela somente se ela ainda não existir.
Preserva tabelas existentes.


In [ ]:
if df_produtos is None:
    print("Sem dados validos. Gravacao ignorada.")

elif spark.catalog.tableExists(tabela_produtos):
    print("Tabela existente preservada:", tabela_produtos)
    display(spark.table(tabela_produtos).limit(15))

else:
    (
        df_produtos.write
        .format("delta")
        .mode("errorifexists")
        .saveAsTable(tabela_produtos)
    )

    print("Tabela Delta criada:", tabela_produtos)
    display(spark.table(tabela_produtos))


## 4. Laboratório isolado de Time Travel

A tabela original `dim_produtos` não será modificada.

O exemplo usa outra tabela, exclusiva do laboratório.
Execute as próximas células em ordem.


In [ ]:
if not spark.catalog.tableExists(tabela_produtos):
    print("Crie primeiro a tabela de produtos.")
else:
    spark.sql(f'''
        CREATE TABLE IF NOT EXISTS {tabela_lab}
        USING DELTA
        AS SELECT * FROM {tabela_produtos}
    ''')

    print("Laboratorio disponivel:", tabela_lab)

    display(spark.sql(f"DESCRIBE HISTORY {tabela_lab}"))

    versao_base = spark.sql(
        f"DESCRIBE HISTORY {tabela_lab}"
    ).selectExpr("max(version) AS versao").first()["versao"]

    print("Versao anterior ao teste:", versao_base)


## 5. Simular erro somente no laboratório

O preço de um único produto será alterado para zero.
A tabela principal não será afetada.


In [ ]:
if "versao_base" not in globals():
    print("Execute a criacao da tabela de laboratorio primeiro.")
else:
    spark.sql(f'''
        UPDATE {tabela_lab}
        SET preco_reais = CAST(0 AS DECIMAL(10,2))
        WHERE id_produto = 'prod_101'
    ''')

    print("Alteracao de teste realizada no laboratorio.")
    display(spark.table(tabela_lab).limit(15))


## 6. Consultar o histórico e restaurar

A versão anterior foi capturada antes da alteração.

A restauração afeta apenas a tabela de laboratório.


In [ ]:
if "versao_base" not in globals():
    print("Execute primeiro as etapas do laboratorio.")
else:
    print("Historico antes da restauracao:")
    display(spark.sql(f"DESCRIBE HISTORY {tabela_lab}"))

    try:
        spark.sql(
            f"RESTORE TABLE {tabela_lab} VERSION AS OF {versao_base}"
        )

        print("Laboratorio restaurado.")
        display(spark.table(tabela_lab).limit(15))

    except Exception as erro:
        print("Restauracao nao concluida.")
        print("Verifique permissoes e retencao Delta.")
        print(type(erro).__name__)